# List expressions

**Advanced Computational Physics** &middot; Master in Physics &middot; USC

*J. A. Hernando Morata*. Original material (2020) in collaboration with
*G. Martínez-Lema* and *M. Kekic*.

**Objectives**

* know what **functional programming** expects from a function;
* walk several lists at once with `zip` and `enumerate`;
* write **comprehensions**: lists, dicts and sets in one expression;
* sort and **reduce** lists: `sorted`, `sum`, `max`, `any`, `all`, `reduce`.

In [ ]:
import math
import random
import numpy as np

## 1. Functional programming

**Functional programming** (FP) builds a program as a flow of data through functions.
The ideal function is **pure**:

* its output depends **only** on its arguments: same input, same output;
* it has **no side effects**: it does not modify its arguments, global variables or
  files, and it does not print.

Data does not change state: it flows from one function to the next.
Pure functions are easy to **test**, to **reuse** and to **combine**.

<div class="admonition important">
<p class="admonition-title"><b>[=] Commit — Pure or not?</b></p>

Decide for each one: pure or not? Why?

```python
def f(x):
    return 2 * x + 1

def g(xs):
    xs.append(0)
    return len(xs)

offset = 3
def h(x):
    return x + offset

def k(x):
    print(x)
    return x
```

</div>

<details>
<summary><b>Solution</b></summary>

* `f`: **pure**.
* `g`: not pure, it modifies its argument (side effect).
* `h`: not pure, its result depends on the global `offset`: change `offset` and the same
  call gives a different answer.
* `k`: not pure, printing is a side effect. Harmless here, but a function that writes to a
  file or a database is not.

A program cannot be 100 % pure: at some point it must read data and write results.
The FP advice is to **push the impure parts to the edges** and keep the core pure.

</details>

Python is not a pure functional language, but it supports the style well:
functions as objects (previous chapter), **list expressions** (this chapter), and
**iterators and generators** (next chapter).

## 2. Two useful built-ins

### 2.1 zip

`zip` walks several iterables **at the same time**, yielding tuples with one item from each:

In [ ]:
names = ['e', 'mu', 'tau']
masses = [0.511, 105.66, 1776.9]       # MeV

for name, mass in zip(names, masses):
    print(f'{name:>4} {mass:8.2f} MeV')

print(list(zip(names, masses)))

<div class="admonition hint">
<p class="admonition-title"><b>[?] What do you see? — Lists of different lengths</b></p>

What does `list(zip([1, 2, 3], ['a', 'b']))` give? Is that a feature or a danger?

</div>

In [ ]:
print(list(zip([1, 2, 3], ['a', 'b'])))

<details>
<summary><b>Solution</b></summary>

`[(1, 'a'), (2, 'b')]`: `zip` stops at the shortest iterable and the `3` is lost, with no
warning. If the lists *must* have the same length (masses and names of particles, `x` and
`y` of a measurement), say it: `zip(xs, ys, strict=True)` raises a `ValueError` when the
lengths differ (Python 3.10+).

</details>

### 2.2 enumerate

`enumerate` gives the **position** and the **item**. Use it instead of `range(len(xs))`:

In [ ]:
for i, name in enumerate(names):
    print(i, name)

<div class="admonition tip">
<p class="admonition-title"><b>[>] Workshop — A phone book</b></p>

1. Given a tuple of names and a tuple of phone numbers, build a dictionary
   `{name: phone}` with `zip`, in one line.
2. Write a function that returns the **position** and the value of the largest item,
   using `enumerate`. Then do it again with `max` and a `key`.

</div>

In [ ]:
# your code here

<details>
<summary><b>Solution</b></summary>

```python
names = ('Ana', 'Luis', 'Eva')
phones = (600111222, 600333444, 600555666)
book = dict(zip(names, phones))

def argmax(xs):
    """Return the position and the value of the largest item of xs."""
    imax, xmax = 0, xs[0]
    for i, x in enumerate(xs):
        if x > xmax:
            imax, xmax = i, x
    return imax, xmax

def argmax2(xs):
    return max(enumerate(xs), key=lambda ix: ix[1])

xs = [3, 9, 2, 9, 1]
assert argmax(xs) == argmax2(xs) == (1, 9)
```

With ties both return the **first** maximum. `np.argmax` does the same.

</details>

## 3. Comprehensions

A **list comprehension** builds a list from an iterable in one expression:

```python
[expression(item) for item in iterable]
[expression(item) for item in iterable if condition(item)]
```

It does what `map` and `filter` do, and it reads almost like mathematics:
$\{x^2 \mid x \in X, \; x \text{ even}\}$.

In [ ]:
xs = list(range(10))

squares = [x * x for x in xs]
evens = [x for x in xs if x % 2 == 0]

print(squares)
print(evens)
print(list(map(lambda x: x * x, xs)) == squares)

### 3.1 Several for

A comprehension can have several `for` and `if`. They are read **in the same order** as
the nested loops:

In [ ]:
xs = [0, 1, 2]
ys = [3, 4, 5]

pairs = [(x, y) for x in xs if x % 2 == 0 for y in ys if y % 2 == 1]
print(pairs)

<div class="admonition important">
<p class="admonition-title"><b>[=] Commit — Write the loops</b></p>

Write the nested loops that build the same `pairs`, **before** opening the solution.
How many items has `[(x, y) for x in range(10) for y in range(20)]`?

</div>

<details>
<summary><b>Solution</b></summary>

```python
pairs = []
for x in xs:
    if x % 2 == 0:
        for y in ys:
            if y % 2 == 1:
                pairs.append((x, y))
```

`10 × 20 = 200`: two `for` make all the combinations, like a double loop. To walk two
lists **in parallel** you need `zip`, not two `for`.

</details>

To walk two lists **in parallel**, use `zip` (not two `for`):

In [ ]:
xs = [0, 1, 2, 3, 4]
ys = [5, 6, 7, 8, 9]
print([y - x for x, y in zip(xs, ys) if x % 2 == 0])

### 3.2 Dict and set comprehensions

The same syntax with `{}` builds a **dictionary** or a **set**:

In [ ]:
masses = {'e': 0.511, 'mu': 105.66, 'tau': 1776.9}

in_gev = {name: m / 1000. for name, m in masses.items()}
light = {name for name, m in masses.items() if m < 200.}

print(in_gev)
print(light)

And with `()`, a **generator expression**: it does not build the list, it produces the
items one by one. Inside a function call the parentheses can be dropped:

In [ ]:
print(sum(x * x for x in range(10)))

We will see what that means, and why it matters, in [Iterators](./about_iterators.ipynb).

<div class="admonition note">
<p class="admonition-title"><b>[!] Discuss — Comprehension or numpy?</b></p>

In physics we usually have arrays of numbers. Compare:

```python
xs = list(range(1_000_000))
squares = [x * x for x in xs]

xa = np.arange(1_000_000)
squares = xa * xa
```

Which one is faster, and by how much? Measure it with `%timeit`. When is a comprehension
still the right tool?

</div>

In [ ]:
xs = list(range(1_000_000))
xa = np.arange(1_000_000)
%timeit [x * x for x in xs]
%timeit xa * xa

<details>
<summary><b>Solution</b></summary>

numpy is one or two orders of magnitude faster: the loop runs in compiled code, on
numbers stored contiguously, without creating a Python object per item.

A comprehension is still the right tool when the items are **not numbers** (strings,
tuples, objects, files), when the list is short, or when the operation has no numpy
equivalent. For numerical arrays, think in numpy.

</details>

## 4. Sorting

`sorted` returns a **new** sorted list; `reverse=True` sorts in **descending** order.

In [ ]:
xs = [8, 10, 9, 5, 3, 1, 4]
print(sorted(xs))
print(sorted(xs, reverse=True))
print(xs)

`key` is a function applied to each item: the items are sorted by its value.

In [ ]:
def distance_to(x0):
    """Return the function x -> |x - x0|."""
    return lambda x: abs(x - x0)

print(sorted(xs, key=distance_to(5)))

In [ ]:
particles = [('e', -1, 0.511), ('p', 1, 938.27), ('mu', -1, 105.66), ('pi', 1, 139.57)]

# by charge, and by mass inside each charge
print(sorted(particles, key=lambda p: (p[1], p[2])))

<div class="admonition hint">
<p class="admonition-title"><b>[?] What do you see? — What went wrong?</b></p>

```python
xs = [3, 1, 2]
xs = xs.sort()
print(xs)
```

</div>

<details>
<summary><b>Solution</b></summary>

It prints `None`. `xs.sort()` sorts the list in place and returns `None`, which is then
assigned to `xs`: the list is lost. Use either `xs.sort()` alone or `xs = sorted(xs)`.

</details>

***[+] Lookout.*** Before `key` existed, sorting used a *comparison* function `cmp(x, y)` returning
`-1, 0, 1`. It still works through `functools.cmp_to_key`, but a `key` is simpler and
faster. Python's sort is **stable**: equal items keep their order, so you can also sort in
several passes (first by the secondary key, then by the primary one).

## 5. Reducing a list

Some built-ins **reduce** an iterable to one value: `sum`, `max`, `min`, and for booleans
`any` (is one of them true?) and `all` (are all of them true?).

In [ ]:
xs = [3, 9, 2, 7]
print(sum(xs), max(xs), min(xs))
print(any(x > 8 for x in xs), all(x > 1 for x in xs))

<div class="admonition important">
<p class="admonition-title"><b>[=] Commit — The empty list</b></p>

**Decide**: what do `sum([])`, `any([])`, `all([])` and `max([])` give?

</div>

In [ ]:
print(sum([]), any([]), all([]))

In [ ]:
max([])

<details>
<summary><b>Solution</b></summary>

`0`, `False` and `True`; `max([])` raises a `ValueError`. Each result is the **neutral
element** of the operation: adding nothing gives 0; "some item is true" is false if there
are no items; "all items are true" cannot be contradicted by an empty list. The maximum
of nothing has no sensible value, so `max` fails (or use `max(xs, default=...)`).

</details>

`functools.reduce(f, xs)` folds the list with any function of two arguments:
`f(f(f(x0, x1), x2), x3)`. The module `operator` has the operators as functions:
`add`, `sub`, `mul`, `truediv`, `pow`, `eq`, `lt`…

In [ ]:
from functools import reduce
from operator import mul

print(reduce(mul, range(1, 6)))     # 5!

<div class="admonition hint">
<p class="admonition-title"><b>[?] What do you see? — Our old functions, again</b></p>

Read these new versions of `polynomial` and of the table of factorials. Are they easier
to read than the loops of the previous chapter? What does each piece do?

```python
def polynomial(coefficients):
    return lambda x: sum(a * x ** i for i, a in enumerate(coefficients))

factorials = [math.factorial(n) for n in range(10)]
```

</div>

## 6. Practice

<div class="admonition tip">
<p class="admonition-title"><b>[>] Workshop — Uniform random numbers</b></p>

Generate $n$ random numbers uniform in $[-L/2, L/2]$ with `random.uniform`, using a
comprehension. Compute their mean and standard deviation with `sum` and comprehensions,
and compare with the expected values, $0$ and $L/\sqrt{12}$.

</div>

In [ ]:
# your code here

<details>
<summary><b>Solution</b></summary>

```python
import random

L, n = 2., 100_000
xs = [random.uniform(-L / 2, L / 2) for _ in range(n)]
mean = sum(xs) / n
std = math.sqrt(sum((x - mean) ** 2 for x in xs) / n)
print(mean, std, L / math.sqrt(12))
```

How close must `mean` be to 0? Its error is $\sigma/\sqrt{n} \approx 0.002$ here: a test
`assert abs(mean) < 5 * std / math.sqrt(n)` is honest; `assert mean == 0` is not.

</details>

<div class="admonition tip">
<p class="admonition-title"><b>[&] Collaborate — Points in the plane</b></p>

Generate $n$ points $(x, y)$ with $x, y$ normally distributed, mean 0 and $\sigma = 1$
(`random.gauss`).
Split the work in the group:

* **A**: the distance of each point to the origin;
* **B**: the points sorted by distance to the origin;
* **C**: the points in the first quadrant at a distance larger than 1.

Use only comprehensions and built-ins. Then each member writes the same with numpy and
checks a teammate's result.

</div>

In [ ]:
# your code here

<details>
<summary><b>Solution</b></summary>

```python
import random

n = 1000
points = [(random.gauss(0., 1.), random.gauss(0., 1.)) for _ in range(n)]

distances = [math.hypot(x, y) for x, y in points]                       # A
by_distance = sorted(points, key=lambda p: math.hypot(*p))              # B
selected = [(x, y) for x, y in points
            if x > 0 and y > 0 and math.hypot(x, y) > 1]                 # C

# check with numpy
pa = np.array(points)
r = np.hypot(pa[:, 0], pa[:, 1])
assert np.allclose(distances, r)
assert np.allclose(by_distance, pa[np.argsort(r)])
mask = (pa[:, 0] > 0) & (pa[:, 1] > 0) & (r > 1)
assert len(selected) == mask.sum()
```

The fraction selected should be about $e^{-1/2}/4 \approx 0.15$: why?

</details>

## 7. Summary

<details>
<summary><b>[i] Cheat sheet — List expressions</b></summary>

| | |
|:--|:--|
| `[f(x) for x in xs if c(x)]` | list comprehension (`map` + `filter`) |
| `{k: v for k, v in d.items()}` | dict comprehension |
| `{f(x) for x in xs}` | set comprehension |
| `sum(f(x) for x in xs)` | generator expression |
| `zip(xs, ys, strict=True)` | walk in parallel; `strict` checks the lengths |
| `enumerate(xs)` | position and item |
| `sorted(xs, key=f, reverse=True)` | new sorted list; `xs.sort()` sorts in place and returns `None` |
| `sum`, `max`, `min`, `any`, `all` | reductions; `max(xs, default=v)` |
| `functools.reduce(f, xs)` | fold with any function of two arguments |

A **pure** function: same input, same output, no side effects.

</details>

### Check yourself

1. Is a function that reads a global variable pure? Why?
2. What does `zip` do with lists of different lengths?
3. How many items does `[(x, y) for x in range(3) for y in range(4)]` have? And
   `[(x, y) for x, y in zip(range(3), range(4))]`?
4. What does `all([])` return, and why?
5. Why is `xs = xs.sort()` a bug?

<details>
<summary><b>Solutions</b></summary>

1. No: its result depends on something that is not an argument and that can change.
2. It stops at the shortest one, silently, unless `strict=True`.
3. 12 (all the combinations) and 3 (in parallel, up to the shortest).
4. `True`: no item contradicts "all are true" (the neutral element of `and`).
5. `sort` works in place and returns `None`, which overwrites the list.

</details>

### Bibliography

1. H. Abelson, G. J. Sussman and J. Sussman, *Structure and Interpretation of Computer
   Programs*, 2nd ed., MIT Press (1996).
   [Online](https://mitpress.mit.edu/sites/default/files/sicp/full-text/book/book.html).
2. [Functional Programming HOWTO](https://docs.python.org/3/howto/functional.html),
   Python documentation.